# Train MobileNetV3 — Standalone (không KD)

Train `FaceRecognitionMobileNetV3` trực tiếp với **MagFace loss**, không dùng teacher hay KD.

| | Model |
|---|---|
| Architecture | `FaceRecognitionMobileNetV3` |
| Backbone | `mobilenetv3_large_100` |
| Params | ~3.6M |
| Loss | `WeightClassMagLoss` (MagFace) |

**Loss:**
```
L_total = L_MagFace(student)
```

**Cách dùng:**
1. Mount Google Drive (cell 1)
2. Sửa `CONFIGURATION` (cell Config)
3. Chạy từ trên xuống

## 1. Mount Drive & Setup môi trường

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

REPO_URL    = 'https://github.com/NguyenXuanBinh22/DATN.git'
REPO_BRANCH = 'convnext-v2-dev'
REPO_DIR    = '/content/FR_Photometric_Stereo'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull origin {REPO_BRANCH}')
    print('Repo đã tồn tại, đã pull latest.')

%cd {REPO_DIR}
print(f'Working dir: {os.getcwd()}')

os.system('pip install -q albumentations==1.3.1 timm tabulate termcolor')

Mounted at /content/drive
/content/FR_Photometric_Stereo
Working dir: /content/FR_Photometric_Stereo


0

## 2. Imports & Cấu hình

In [ ]:
%cd /content/FR_Photometric_Stereo
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import albumentations as A
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.utils.tensorboard import SummaryWriter
from tabulate import tabulate

from going_modular.dataloader.multitask import create_multitask_datafetcher, create_eval_loaders
from going_modular.model.FaceRecognitionMobileNetV3 import FaceRecognitionMobileNetV3
from going_modular.loss.WeightClassMagLoss import WeightClassMagLoss
from going_modular.utils.transforms import RandomResizedCropRect, GaussianNoise
from going_modular.utils.roc_auc_id import (
    compute_id_auc, compute_rank1,
    compute_id_auc_gallery_probe, compute_rank1_gallery_probe,
)
from going_modular.utils.MultiMetricEarlyStopping import MultiMetricEarlyStopping
from going_modular.utils.ModelCheckPoint import ModelCheckpoint
from going_modular.utils.ExperimentManager import ExperimentManager

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

/content/FR_Photometric_Stereo
Device: cpu


In [ ]:
# ════════════════════════════════════════════════════════════
#  CẤU HÌNH — chỉnh sửa ở đây
# ════════════════════════════════════════════════════════════

DRIVE_DATASET_DIR = '/content/drive/MyDrive/Photometric_DB_Full/'

EXPERIMENT_NAME = 'MobileNetV3_Standalone_Albedo_29'

CONFIGURATION = {
    'note':        EXPERIMENT_NAME,
    'dataset_dir': DRIVE_DATASET_DIR,

    'output_dir':  '/content/drive/MyDrive/',

    # Modality: 'albedo' | 'normalmap' | 'depthmap'
    'type':        'albedo',

    'backbone':    'mobilenetv3_large_100',

    'use_sampler': True,
    'device':      device,
    'epochs':      40,
    'batch_size':  32,
    'image_size':  112,
    'base_lr':     1e-4,
    'num_classes': None,   # tự động lấy từ CSV
}

print(f"Dataset dir : {CONFIGURATION['dataset_dir']}")
print(f"Output dir  : {CONFIGURATION['output_dir']}")

Dataset dir : /content/drive/MyDrive/Photometric_DB_Full/
Output dir  : /content/drive/MyDrive/


## 3. Data Loading

In [ ]:
dataset_dir = CONFIGURATION['dataset_dir']

train_csv = os.path.join(dataset_dir, 'train_split.csv')
if not os.path.exists(train_csv):
    train_csv = os.path.join(dataset_dir, 'dataset', 'train_split.csv')
if not os.path.exists(train_csv):
    train_csv = os.path.join(dataset_dir, 'train_set.csv')
if not os.path.exists(train_csv):
    raise FileNotFoundError(
        f'Không tìm thấy CSV train tại {dataset_dir}.\n'
        f'Kiểm tra lại DRIVE_DATASET_DIR.'
    )
print(f'Train CSV: {train_csv}')

df_train = pd.read_csv(train_csv)
CONFIGURATION['num_classes'] = int(df_train['id'].nunique())
print(f'num_classes : {CONFIGURATION["num_classes"]}')
print(f'Số mẫu train: {len(df_train)}')

train_transform = A.Compose([
    RandomResizedCropRect(CONFIGURATION['image_size']),
    GaussianNoise(p=0.2),
    A.HorizontalFlip(p=0.5),
])
test_transform = A.Compose([
    A.Resize(CONFIGURATION['image_size'], CONFIGURATION['image_size']),
])

train_dl, test_dl, _ = create_multitask_datafetcher(
    CONFIGURATION, train_transform, test_transform, 'train_split.csv', 'probe_split.csv'
)
print(f'Train batches: {len(train_dl)} | Test batches (probe): {len(test_dl)}')

gallery_dl, probe_dl = create_eval_loaders(CONFIGURATION, test_transform)

Train CSV: /content/drive/MyDrive/Photometric_DB_Full/train_split.csv
num_classes : 352
Số mẫu train: 2622
>>> SingleLoader: MODE = PK SAMPLER
Train batches: 81 | Test batches (probe): 9
Gallery: 68 ảnh | Probe: 288 ảnh
Shared identity space: 68 identities


## 4. Model

In [ ]:
model = FaceRecognitionMobileNetV3(
    num_classes=CONFIGURATION['num_classes'],
    backbone=CONFIGURATION['backbone'],
)
model.to(device)

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total_p:,}')
print(f'Trainable params: {trainable_p:,}')

with torch.no_grad():
    _dummy = torch.randn(2, 3, 112, 112).to(device)
    _emb = model.get_embedding(_dummy)
    print(f'Embedding shape: {_emb.shape}')   # [2, 512]

Total params    : 3,645,744
Trainable params: 3,645,744
Embedding shape: torch.Size([2, 512])


## 5. Loss

```
L_total = L_MagFace
```

In [ ]:
criterion = WeightClassMagLoss(file_path=train_csv)
print('WeightClassMagLoss khởi tạo thành công.')

WeightClassMagLoss khởi tạo thành công.


## 6. Training

In [ ]:
def train_epoch(train_dl, model, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for X, y in train_dl:
        X, y = X.to(device), y.to(device)
        id_labels = y[:, 0]

        logits, norm = model(X)
        loss = criterion(logits, id_labels, norm)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_dl)


def display_metrics(epoch, train_metrics, test_metrics):
    rows = []
    for k in train_metrics:
        tv = train_metrics[k]
        ev = test_metrics.get(k, '-')
        fmt = lambda v: f'{v:.4f}' if isinstance(v, float) else str(v)
        rows.append([k, fmt(tv), fmt(ev)])
    print(f'\nEp {epoch}:')
    print(tabulate(rows, headers=['Metric', 'Train', 'Test'], tablefmt='fancy_grid'))

In [ ]:
optimizer = Adam(model.parameters(), lr=CONFIGURATION['base_lr'])
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)

manager = ExperimentManager(CONFIGURATION)
manager.log_text(
    f"Model: {CONFIGURATION['backbone']} | "
    f"Modality: {CONFIGURATION['type']} | "
    f"Loss: MagFace (standalone)"
)

ckpt_saver = ModelCheckpoint(
    output_dir=manager.ckpt_dir,
    mode='max',
    best_metric_name='auc_id_cosine',
)
early_stopping = MultiMetricEarlyStopping(
    monitor_keys=['auc_id_cosine'],
    patience=5,
    mode='max',
    verbose=1,
    save_dir=manager.ckpt_dir,
    start_from_epoch=5,
)

writer = SummaryWriter(log_dir=manager.log_dir)
print(f'Experiment dir: {manager.exp_dir}')
print(f'Checkpoint dir: {manager.ckpt_dir}')

KHOI TAO THI NGHIEM: MobileNetV3_Standalone_Albedo_29
Luu tru tai: /content/drive/MyDrive/experiments/MobileNetV3_Standalone_Albedo_29
Thoi gian: 2026-06-29 04:06:01
--------------------------------------------------
Model: mobilenetv3_large_100 | Modality: albedo | Loss: MagFace (standalone)
Experiment dir: /content/drive/MyDrive/experiments/MobileNetV3_Standalone_Albedo_29
Checkpoint dir: /content/drive/MyDrive/experiments/MobileNetV3_Standalone_Albedo_29/checkpoints


In [ ]:
START_EPOCH = 0

manager.log_text('BAT DAU TRAINING STANDALONE')

for epoch in range(START_EPOCH, CONFIGURATION['epochs']):
    manager.log_text(f'\n--- Epoch {epoch+1}/{CONFIGURATION["epochs"]} ---')

    train_loss = train_epoch(train_dl, model, criterion, optimizer, device)

    train_auc = compute_id_auc(train_dl, model, device)
    test_auc  = compute_id_auc(test_dl,  model, device)

    train_metrics = {
        'loss':             train_loss,
        'auc_id_cosine':    train_auc['id_cosine'],
        'auc_id_euclidean': train_auc['id_euclidean'],
    }
    test_metrics = {
        'auc_id_cosine':    test_auc['id_cosine'],
        'auc_id_euclidean': test_auc['id_euclidean'],
    }

    writer.add_scalar('Loss/total', train_loss, epoch + 1)
    writer.add_scalars('AUC/cosine',
        {'train': train_auc['id_cosine'],    'test': test_auc['id_cosine']},    epoch + 1)
    writer.add_scalars('AUC/euclidean',
        {'train': train_auc['id_euclidean'], 'test': test_auc['id_euclidean']}, epoch + 1)

    display_metrics(epoch + 1, train_metrics, test_metrics)
    manager.log_metrics(epoch + 1, {**train_metrics, **test_metrics})

    ckpt_saver(model, optimizer, epoch + 1, test_metrics, scheduler)

    # Save checkpoint at specific epochs
    if (epoch + 1) in [10, 20, 30]:
        epoch_ckpt_path = os.path.join(manager.ckpt_dir, f'epoch_{epoch+1}_model.pth')
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'metrics': test_metrics
        }, epoch_ckpt_path)
        # manager.log_text(f'Checkpoint saved at epoch {epoch+1}: {epoch_ckpt_path}')

    early_stopping(test_metrics, model, epoch + 1)
    scheduler.step(epoch)

    if early_stopping.early_stop:
        manager.log_text('Early stopping triggered.')
        break

writer.close()
manager.log_text('TRAINING HOAN TAT.')

BAT DAU TRAINING STANDALONE

--- Epoch 1/40 ---

Ep 1:
╒══════════════════╤═════════╤════════╕
│ Metric           │   Train │ Test   │
╞══════════════════╪═════════╪════════╡
│ loss             │ 29.5785 │ -      │
├──────────────────┼─────────┼────────┤
│ auc_id_cosine    │  0.9188 │ 0.8712 │
├──────────────────┼─────────┼────────┤
│ auc_id_euclidean │  0.9188 │ 0.8712 │
╘══════════════════╧═════════╧════════╛
Ep 1: loss: 29.5785, auc_id_cosine: 0.8712, auc_id_euclidean: 0.8712
--> SAVE BEST MODEL (auc_id_cosine: 0.8712)

--- Epoch 2/40 ---

Ep 2:
╒══════════════════╤═════════╤════════╕
│ Metric           │   Train │ Test   │
╞══════════════════╪═════════╪════════╡
│ loss             │ 31.8148 │ -      │
├──────────────────┼─────────┼────────┤
│ auc_id_cosine    │  0.9276 │ 0.8841 │
├──────────────────┼─────────┼────────┤
│ auc_id_euclidean │  0.9276 │ 0.8841 │
╘══════════════════╧═════════╧════════╛
Ep 2: loss: 31.8148, auc_id_cosine: 0.8841, auc_id_euclidean: 0.8841
--> SAVE BEST MO

## 7. Resume Training từ Checkpoint

> Chạy cell này khi Colab disconnect và muốn tiếp tục train.  
> **Cách dùng:** Chạy Setup → Imports → Data → Model → Loss → Setup Train,  
> sau đó chạy cell này, rồi chạy lại cell fit.

In [ ]:
# CKPT_PATH = os.path.join(manager.ckpt_dir, 'last_model.pth')

# if not os.path.exists(CKPT_PATH):
#     raise FileNotFoundError(f'Không tìm thấy checkpoint: {CKPT_PATH}')

# checkpoint = torch.load(CKPT_PATH, map_location=device)
# model.load_state_dict(checkpoint['model_state_dict'])
# optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
# if 'scheduler_state_dict' in checkpoint:
#     scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

# START_EPOCH = checkpoint['epoch']
# print(f'Resume từ epoch {START_EPOCH} — chạy lại cell "cell-fit" để tiếp tục.')

## 8. Đánh giá Final

In [ ]:
best_ckpt_path = os.path.join(manager.ckpt_dir, 'best_model.pth')
best_ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(best_ckpt['model_state_dict'])
model.eval()


gp_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, model, device)
gp_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, model, device)

rows = [
    ['Cosine AUC    (gallery→probe)', f"{gp_auc['id_cosine']:.4f}"],
    ['Euclidean AUC (gallery→probe)', f"{gp_auc['id_euclidean']:.4f}"],
    ['Rank-1 Acc    (gallery→probe)', f"{gp_rank1:.4f}"],
]
print(f"\nMobileNetV3 Standalone — backbone: {CONFIGURATION['backbone']}")
print(tabulate(rows, headers=['Metric', 'Value'], tablefmt='fancy_grid'))


MobileNetV3 Standalone — backbone: mobilenetv3_large_100
╒═══════════════════════════════╤═════════╕
│ Metric                        │   Value │
╞═══════════════════════════════╪═════════╡
│ Cosine AUC    (gallery→probe) │  0.9383 │
├───────────────────────────────┼─────────┤
│ Euclidean AUC (gallery→probe) │  0.9383 │
├───────────────────────────────┼─────────┤
│ Rank-1 Acc    (gallery→probe) │  0.6701 │
╘═══════════════════════════════╧═════════╛


## 9. Export ONNX

In [ ]:
# class InferenceWrapper(nn.Module):
#     """Backbone + embedding + L2 normalize — không có MagLinear."""
#     def __init__(self, model):
#         super().__init__()
#         self.backbone  = model.backbone
#         self.embedding = model.embedding

#     def forward(self, x):
#         emb = self.embedding(self.backbone(x))
#         return F.normalize(emb, p=2, dim=1)


# inference_model = InferenceWrapper(model).eval().cpu()
# dummy_input = torch.randn(1, 3, 112, 112)

# onnx_path = os.path.join(manager.ckpt_dir, 'mobilenetv3_standalone_fr.onnx')

# torch.onnx.export(
#     inference_model,
#     dummy_input,
#     onnx_path,
#     input_names=['input'],
#     output_names=['embedding'],
#     dynamic_axes={'input': {0: 'batch'}, 'embedding': {0: 'batch'}},
#     opset_version=17,
# )
# print(f'Exported ONNX: {onnx_path}')